### 1. Survey Combiner
##### This script combines all essential / indoor-vital data and executes the methods outline in the flow charts perscribed in the data methods folder. Throughout the script exerpts are commented out but can be uncommeneted if you wish to see the data at that stage.

In [ ]:
import pandas as pd
import country_converter as coco
import sys
import os
from pathlib import Path
cc = coco.CountryConverter()

# create file paths
cwd = Path.cwd()
data_folder = cwd.parent / "data"
isco_08_poll_binary_PATH = data_folder / "ISCO-08 OpinionPollCensus.xlsx"
onet_context_by_soc_code_PATH = data_folder / "Indoors_Environmentally_Controlled_data.csv"
soc_isco_crosswalk_PATH = data_folder / "ISCO_SOC_Crosswalk.csv"
ilo_employment_by_isco_08_l2_PATH = data_folder / "ILO_ISCO_08_GLB.csv"
world_bank_labour_force_by_country_PATH = data_folder / "LFData_WB_plus.xlsx"



In [ ]:
# import required data
isco_08_poll_binary_df = pd.read_excel(isco_08_poll_binary_PATH, engine="openpyxl")
onet_context_by_soc_code_df = pd.read_csv(onet_context_by_soc_code_PATH)
soc_isco_crosswalk_df = pd.read_csv(soc_isco_crosswalk_PATH)
ilo_employment_by_isco_08_l2_df = pd.read_csv(ilo_employment_by_isco_08_l2_PATH)
labourForce_df = pd.read_excel(world_bank_labour_force_by_country_PATH, usecols=[0, 1, 3])


In [ ]:
def normalise_country_keys(d):
    return {cc.convert(names=k, to='name_short', not_found='not found'): v for k, v in d.items()}
# helpful function that make country names standard, when the country name is the key in the dict

#### 1.1Reading/Processing ISCO-08 Poll results from the complete excel (found in the data folder)

In [ ]:
isco_08_poll_binary_df = isco_08_poll_binary_df[['ISCO-08','Census']]
isco_08_poll_binary_df['ISCO-08'] = isco_08_poll_binary_df['ISCO-08'].astype(str).str.zfill(4)
isco_08_poll_binary_df = isco_08_poll_binary_df.set_index('ISCO-08')


In [ ]:
#isco_08_poll_binary_df

#### 1.2 Reading/Processing ONET data on Indoor/Outdoor Context (found in the Data folder)

In [ ]:
onet_context_by_soc_code_df = onet_context_by_soc_code_df[['Context','Code']] # only keep nessesary columns
# NO ARMY INCLUDED, Use Blue Print results (%40 of Army considered essential/vital)

#### 1.3 Reading/Processing SOC - ISCO-08 crosswalk, i.e. the map between codes (found in the Data folder)

In [ ]:
soc_isco_crosswalk_df = soc_isco_crosswalk_df[['2010 SOC Code','ISCO-08 Code']]
soc_isco_crosswalk_df['ISCO-08 Code'] = soc_isco_crosswalk_df['ISCO-08 Code'].astype(str)
# '''Bureau of Labor Statistics,,,,,
# On behalf of the Standard Occupational Classification Policy Committee (SOCPC),,,,,
# ,,,,,
# August 2012 (Updated June 2015),,,,,
# Questions should be emailed to soc@bls.gov'''
# note some are mapped to level 3 ISCO-08 codes, hence it should be truncated

In [ ]:
#soc_isco_crosswalk_df

In [ ]:
soc_isco_crosswalk_dict = soc_isco_crosswalk_df.set_index('2010 SOC Code')['ISCO-08 Code'].to_dict()

In [ ]:
onet_context_by_soc_code_df['Code'] = onet_context_by_soc_code_df['Code'].astype(str).str.split('.').str[0]
onet_context_by_soc_code_df['Context'] = onet_context_by_soc_code_df['Context']/100
onet_context_by_soc_code_dict = onet_context_by_soc_code_df.set_index('Code')['Context'].to_dict()

In [ ]:
ISCO_CONTEXT_InOutdoor_dict = {}
ONET_CODES_NOT_IN_MAP_dict = {} # if the Onet codes is not in the SOC Map it will be attached to a list of codes in the same 2-digit SOC group and added to the ISCO-08 2 digit codes further on
for k , v in onet_context_by_soc_code_dict.items():
  try:
    if k in ISCO_CONTEXT_InOutdoor_dict.keys(): soc_isco_crosswalk_dict[k].append(v)
    else: ISCO_CONTEXT_InOutdoor_dict[soc_isco_crosswalk_dict[k]] = [v]
  except:
    if k[:2] not in ONET_CODES_NOT_IN_MAP_dict.keys(): ONET_CODES_NOT_IN_MAP_dict[k[:2]] = [v]
    else: ONET_CODES_NOT_IN_MAP_dict[k[:2]].append(v)
    for i,v in soc_isco_crosswalk_dict.items():
      zone_list = []
      if k[:2] in i:
        zone_list.append(v)

L2_SOC_L2_ISCO_map = {}
for k in soc_isco_crosswalk_dict.keys():
  if k[:2] in ONET_CODES_NOT_IN_MAP_dict.keys():
    if k[:2] not in L2_SOC_L2_ISCO_map.keys(): L2_SOC_L2_ISCO_map[k[:2]] = [soc_isco_crosswalk_dict[k][:2]]
    else: L2_SOC_L2_ISCO_map[k[:2]].append(soc_isco_crosswalk_dict[k][:2])

L2_ISCO_MISSING = {}
for k, v in ONET_CODES_NOT_IN_MAP_dict.items():
  L2_ISCO_MISSING[max(L2_SOC_L2_ISCO_map[k], key=L2_SOC_L2_ISCO_map[k].count)] = v

In [ ]:
ISCO_CONTEXT_InOutdoor_df = pd.DataFrame.from_dict(ISCO_CONTEXT_InOutdoor_dict, orient='index', columns=['Context'])
ISCO_CONTEXT_InOutdoor_df.index.name = 'ISCO-08 Code'

#### 1.4 Collapse the Indoor / Outdoor Context Data to the Level 2 ISCO codes


In [ ]:
abs_level = 2  # Truncated to the 2 digit level

# Ensure codes are of the same format before mereg
L = isco_08_poll_binary_df.copy()
R = ISCO_CONTEXT_InOutdoor_df.copy()
L.index = L.index.astype(str).str.zfill(4)
R.index = R.index.astype(str).str.zfill(4)

# Merge
df = pd.merge(L, R, left_index=True, right_index=True, how='left')

# Clean up null errors
df['Context'] = pd.to_numeric(df.get('Context'), errors='coerce')
df['Census']  = pd.to_numeric(df.get('Census'),  errors='coerce')

# Collapse by the mean of its level, if not take the mean of the level above
df['_L3'] = df.index.str[:3]
df['_L2'] = df.index.str[:2]
df['_L1'] = df.index.str[:1]

mean_L3  = df.groupby('_L3')['Context'].transform('mean')
mean_L2  = df.groupby('_L2')['Context'].transform('mean')
mean_L1  = df.groupby('_L1')['Context'].transform('mean')
mean_all = df['Context'].mean()

df['Context Proj'] = (
    df['Context']
      .fillna(mean_L3)
      .fillna(mean_L2)
      .fillna(mean_L1)
)

# Manual overrides
overrides = ['0110', '0210', '0310']
df.loc[overrides, 'Context Proj'] = 1

# Tidy Up
df.drop(columns=['_L3','_L2','_L1', 'Context'], inplace=True)

ISCO_LVL4_WEIGHTS = df

In [ ]:
#ISCO_LVL4_WEIGHTS

In [ ]:
abs_level = 2  # Truncte to the 2 digit level

df = ISCO_LVL4_WEIGHTS.copy()
df.index = df.index.astype(str).str.zfill(4)

# Group to level 2
ISCO_LVL2_WEIGHTS = (
    df.assign(ISCO_trunc=df.index.str[:abs_level])
      .groupby('ISCO_trunc')
      .agg({
          'Census': 'mean',
          'Context Proj': 'mean'
      })
      .rename_axis('ISCO-08')
      .rename(columns={'Census': 'Vital Weight POLL'})
)

# Add Essential Weights from the ILO paper for the upper bound.
# Binary 1 / 0 rather then continuous values
ILO_LVL2_ESSENTIAL_GROUPS = [61,62,63,92,94,22,32,53,52,95,54,71,72,73,74,75,81,82,93,91,96,83,31,44,51,1,2,3]

# Ensure entrys are integers
ISCO_LVL2_WEIGHTS['Essential Weight ILO'] = (
    ISCO_LVL2_WEIGHTS.index.astype(int)
    .isin(ILO_LVL2_ESSENTIAL_GROUPS)
    .astype(int)
)


In [ ]:
#ISCO_LVL2_WEIGHTS

In [ ]:
farms = ISCO_LVL2_WEIGHTS.loc['62'] # Extrapolate all farming related roles from the single farming,
ISCO_LVL2_WEIGHTS.loc['61'] = farms # as the Indoor / Outdoor context is missing this data point
# Subsistence farmers (63) are 100% outdoor (Context Proj = 0). They are still
# essential per ILO Table A2 (Food systems group), so leave Essential Weight ILO
# and Vital Weight POLL unchanged - only zero the indoors fraction.
ISCO_LVL2_WEIGHTS.at['63', 'Context Proj'] = 0

In [ ]:
# Drop Poll-essential L2 codes that ILO's methodology (Table A2) classified as
# teleworkable and therefore excluded from key workers. The team's poll marks a
# handful of L4 codes within these L2 groups as vital, but we set the L2 weight
# to 0 to keep Vital strictly within ILO's non-teleworkable occupations.
NON_ILO_POLL_CODES = ['13', '21', '33', '35']
for code in NON_ILO_POLL_CODES:
    if code in ISCO_LVL2_WEIGHTS.index:
        ISCO_LVL2_WEIGHTS.at[code, 'Vital Weight POLL'] = 0

# ILO Table A2 occupational groups (+ 9th armed-forces group). Codes not in any
# of these groups have Essential Weight ILO == 0 already, so a missing/zero
# Group Overlap is harmless.
ISCO_L2_TO_GROUP = {
    '61': 'Food', '62': 'Food', '63': 'Food', '92': 'Food', '94': 'Food',
    '22': 'Health', '32': 'Health', '53': 'Health',
    '52': 'Retail', '95': 'Retail',
    '54': 'Security',
    '71': 'Manual', '72': 'Manual', '73': 'Manual', '74': 'Manual', '75': 'Manual',
    '81': 'Manual', '82': 'Manual', '93': 'Manual',
    '91': 'Cleaning', '96': 'Cleaning',
    '83': 'Transport',
    '31': 'Tech', '44': 'Tech', '51': 'Tech',
    '01': 'ArmedForces', '02': 'ArmedForces', '03': 'ArmedForces',
}
# Per-group key-sector overlap from ILO Figure A1 (intersection of essential
# occupations with essential ISIC sectors). Armed forces sit outside the 8 ILO
# groups so we use the existing 40% Blue Print assumption.
GROUP_OVERLAP = {
    'Food': 0.895,
    'Health': 0.819,
    'Retail': 0.876,
    'Security': 0.846,
    'Transport': 0.869,
    'Manual': 0.335,
    'Cleaning': 0.485,
    'Tech': 0.320,
    'ArmedForces': 0.40,
}
ISCO_LVL2_WEIGHTS['Group'] = ISCO_LVL2_WEIGHTS.index.map(ISCO_L2_TO_GROUP)
ISCO_LVL2_WEIGHTS['Group Overlap'] = (
    ISCO_LVL2_WEIGHTS['Group'].map(GROUP_OVERLAP).fillna(0.0)
)

# Apply Group Overlap to both indoors (Context Proj) and total weights so the
# ILO-Figure-A1 sector reduction propagates through indoor counts and through the totals.
ISCO_LVL2_WEIGHTS["ISCO_08_PollWeights"] = (
    ISCO_LVL2_WEIGHTS['Vital Weight POLL']
    * ISCO_LVL2_WEIGHTS['Context Proj']
    * ISCO_LVL2_WEIGHTS['Group Overlap']
)
ISCO_LVL2_WEIGHTS["ISCO_08_ILOWeights"] = (
    ISCO_LVL2_WEIGHTS['Essential Weight ILO']
    * ISCO_LVL2_WEIGHTS['Context Proj']
    * ISCO_LVL2_WEIGHTS['Group Overlap']
)

# Total weights (no Context Proj / indoors filter): used for total Vital and Essential worker estimates
ISCO_LVL2_WEIGHTS["ISCO_08_PollWeights_Total"] = (
    ISCO_LVL2_WEIGHTS['Vital Weight POLL']
    * ISCO_LVL2_WEIGHTS['Group Overlap']
)
ISCO_LVL2_WEIGHTS["ISCO_08_ILOWeights_Total"] = (
    ISCO_LVL2_WEIGHTS['Essential Weight ILO']
    * ISCO_LVL2_WEIGHTS['Group Overlap']
)

In [ ]:
#ISCO_LVL2_WEIGHTS

#### 2. The following section extracts the ILO data source and complies a dictionary of countries and the given number of workers employed under each level 2 ISCO-08 code.

In [ ]:
ilo_employment_by_isco_08_l2_df['ISCO-8 L2 Code'] = ilo_employment_by_isco_08_l2_df['classif1.label'].str.split(':').str[1].str[1:4]
ilo_employment_by_isco_08_l2_df['Employment'] = ilo_employment_by_isco_08_l2_df['obs_value'] * 1000
ilo_employment_by_isco_08_l2_df = ilo_employment_by_isco_08_l2_df.rename(columns={'ref_area.label': 'Country'})[['Country', 'ISCO-8 L2 Code', 'Employment', 'time']]

In [ ]:
#ilo_employment_by_isco_08_l2_df

In [ ]:
EmploymentByISOC8 = {}
for _, row in ilo_employment_by_isco_08_l2_df.iterrows():
    c    = row['Country']
    code = row['ISCO-8 L2 Code']
    yr   = row['time']
    emp  = row['Employment']
    EmploymentByISOC8.setdefault(c, {}).setdefault(code, {})[yr] = emp

EmploymentByISOC8 = normalise_country_keys(EmploymentByISOC8)

In [ ]:
#EmploymentByISOC8

In [ ]:
for code_dict in EmploymentByISOC8.values():
    for isco_code, year_dict in list(code_dict.items()):
        if isinstance(year_dict, dict) and year_dict:
            # Filter out NaN values before finding the latest year
            valid_years = {year: value for year, value in year_dict.items() if pd.notna(value)}
            if valid_years:
                latest = max(valid_years)
                code_dict[isco_code] = valid_years[latest]
            else:
                # Handle cases where all values for a given ISCO code are NaN
                code_dict[isco_code] = None # Or some other appropriate value

In [ ]:
# Convert the Series directly to a dictionary
ISCO_08_PollWeights = ISCO_LVL2_WEIGHTS['ISCO_08_PollWeights'].to_dict()
ISCO_08_ILOWeights = ISCO_LVL2_WEIGHTS['ISCO_08_ILOWeights'].to_dict()

# Same dictionaries without the indoors filter applied (used for total Vital / Essential worker estimates)
ISCO_08_PollWeights_Total = ISCO_LVL2_WEIGHTS['ISCO_08_PollWeights_Total'].to_dict()
ISCO_08_ILOWeights_Total = ISCO_LVL2_WEIGHTS['ISCO_08_ILOWeights_Total'].to_dict()


In [ ]:
#ISCO_08_PollWeights

In [ ]:
# Calculate indoor vital employment by multiplying employment by the weights
IVW_dict_Poll = {}
# Total Vital Worker employment (Poll weights without the indoors filter)
VW_dict_Poll = {}

# Iterate through the nested_dict
for country, employment_dict in EmploymentByISOC8.items():
    IVW_dict_Poll[country] = {} # Initialize dictionary for the country
    VW_dict_Poll[country] = {}
    for isco_code, employment in employment_dict.items():
        # Ensure isco_code is a string and strip whitespace
        isco_code_str = str(isco_code).strip()

        # Check if the ISCO code exists in the dictionary and employment is not NaN
        if isco_code_str in ISCO_08_PollWeights.keys() and pd.notna(employment):
            weight = ISCO_08_PollWeights[isco_code_str]
            # Check if both weight and employment are numeric before multiplication
            if pd.notna(weight):
                essential_employment = employment * weight
            else: essential_employment = 0
            IVW_dict_Poll[country][isco_code_str] = essential_employment # Use stripped code as key

            weight_total = ISCO_08_PollWeights_Total[isco_code_str]
            if pd.notna(weight_total):
                vital_employment = employment * weight_total
            else: vital_employment = 0
            VW_dict_Poll[country][isco_code_str] = vital_employment

In [ ]:
# Calculate indoor essential employment by multiplying employment by the weights
IEW_dict_ILO = {}
# Total Essential Worker employment (ILO weights without the indoors filter)
EW_dict_ILO = {}

# Iterate through the nested_dict (which has structure {'Country': {ISCO Code: Employment}})
for country, employment_dict in EmploymentByISOC8.items():
    IEW_dict_ILO[country] = {} # Initialize dictionary for the country
    EW_dict_ILO[country] = {}
    for isco_code, employment in employment_dict.items():
        # Ensure isco_code is a string and strip whitespace for lookup
        isco_code_str = str(isco_code).strip()

        # Check if the ISCO code exists in the dictionary and employment is not NaN
        if isco_code_str in ISCO_08_ILOWeights.keys() and pd.notna(employment):
            weight = ISCO_08_ILOWeights[isco_code_str]
            # Check if both weight and employment are numeric before multiplication
            if pd.notna(weight):
                essential_employment = employment * weight
            else: essential_employment = 0
            IEW_dict_ILO[country][isco_code_str] = essential_employment # Use stripped code as key

            weight_total = ISCO_08_ILOWeights_Total[isco_code_str]
            if pd.notna(weight_total):
                essential_total_employment = employment * weight_total
            else: essential_total_employment = 0
            EW_dict_ILO[country][isco_code_str] = essential_total_employment

In [ ]:
IEW_ILO = {}

for country, isco_data in IEW_dict_ILO.items():
    IEW_ILO[country] = sum(isco_data.values())

EW_ILO = {}

for country, isco_data in EW_dict_ILO.items():
    EW_ILO[country] = sum(isco_data.values())

# Armed-forces (ISCO 01/02/03) sub-sums extracted from the ILO dicts. Kept as
# diagnostic-only series so we can quantify how much they move the totals.
ARMED_FORCES_L2 = ('01', '02', '03')
AF_indoor_essential = {}
AF_essential = {}

for country, isco_data in IEW_dict_ILO.items():
    AF_indoor_essential[country] = sum(isco_data.get(c, 0) for c in ARMED_FORCES_L2)

for country, isco_data in EW_dict_ILO.items():
    AF_essential[country] = sum(isco_data.get(c, 0) for c in ARMED_FORCES_L2)

In [ ]:
IVW_Poll = {}

for country, isco_data in IVW_dict_Poll.items():
    IVW_Poll[country] = sum(isco_data.values())

VW_Poll = {}

for country, isco_data in VW_dict_Poll.items():
    VW_Poll[country] = sum(isco_data.values())

In [ ]:
IVW_Poll_pc = {}
VW_Poll_pc = {}

for country, isco_data in EmploymentByISOC8.items():
    total_employment = isco_data['Tot']
    IVW_Poll_pc[country] = float(IVW_Poll[country]/total_employment)
    VW_Poll_pc[country] = float(VW_Poll[country]/total_employment)

In [ ]:
IEW_ILO_pc = {}
EW_ILO_pc = {}
AF_indoor_essential_pc = {}
AF_essential_pc = {}

for country, isco_data in EmploymentByISOC8.items():
    total_employment = isco_data['Tot']
    IEW_ILO_pc[country] = float(IEW_ILO[country]/total_employment)
    EW_ILO_pc[country] = float(EW_ILO[country]/total_employment)
    AF_indoor_essential_pc[country] = float(AF_indoor_essential.get(country, 0)/total_employment)
    AF_essential_pc[country] = float(AF_essential.get(country, 0)/total_employment)

# 3 Labour Force Data


##### 3.1 Import the most recent figures for labour force by ISO-3 countries. Missing countrys labour forces were manually aquired and there soruces can be found the in the "LFData_WB_plus.xlsx" sheet in the data folder

In [ ]:
labourForce_df['Country Name'] = cc.convert(labourForce_df['Country Name'], to='short_name', not_found='not found')

labourForce_df['Region'] = cc.convert(labourForce_df['Country Name'], to='UNregion', not_found='not found')
#labourForce_df.dropna(subset=['Country Name'], inplace=True) # append a column of the countrys UNregion.

In [ ]:
#labourForce_df

In [ ]:
# labourForce_df:
# the full data frame of Essential / Indoor worker counts by Country from both the Upperbound 
# (strickly essential workers, aquired form the poll) and the Lowerbound
# (Covid-style essential worker estimate, ILO-the value of essential work)

In [ ]:
for idx, row in labourForce_df.iterrows():
    country = row["Country Name"]
    if country in IEW_ILO_pc.keys():
      labourForce_df.at[idx, "%Indoor Essential Workers"] = IEW_ILO_pc[country]
      labourForce_df.at[idx, "%Indoor Vital Workers"] = IVW_Poll_pc[country]
      labourForce_df.at[idx, "%Essential Workers"] = EW_ILO_pc[country]
      labourForce_df.at[idx, "%Vital Workers"] = VW_Poll_pc[country]
      labourForce_df.at[idx, "%Armed Forces (Indoor Essential)"] = AF_indoor_essential_pc[country]
      labourForce_df.at[idx, "%Armed Forces (Essential)"] = AF_essential_pc[country]

###### The following dictionary is used to project missing labour force breakdown data (ISCO-08 breakdown), by taking the average of neighbouring countries with similar workfoces and GDP per Capita. The dictionary may need to be run multiple times if there are interdepenacies within the dictionary

In [ ]:
similar_iso3 = {
    "ABW": ["CUW","SXM","MHL"],
    "AIA": ["VGB","TCA","MAF"],
    "AND": ["LIE","CYP","MCO"],
    "ARM": ["GEO","AZE","ALB"],
    "ASM": ["GUM","MNP","WSM"],
    "ATA": ["ATF","HMD","SGS"],
    "ATF": ["HMD","BVT","SGS"],
    "ATG": ["KNA","MDG","VCT"],
    "AZE": ["GEO","KAZ","UZB"],
    "BES": ["ABW","CUW","SXM"],
    "BHR": ["ARE","QAT","OMN"],
    "BLM": ["MAF","SXM","GLP"],
    "BMU": ["MHL"],
    "BVT": ["ATF","HMD","SGS"],
    "CAF": ["TCD","SSD","NER"],
    "CAN": ["USA"],
    "CCK": ["CXR","NFK","HMD"],
    "CHI": ["GBR"],
    "CHN": ["JPN","IND","VNM"],
    "CMR": ["COG","GAB","NGA"],
    "COG": ["GAB","CMR","GNQ"],
    "COM": ["MDG","MUS","SYC"],
    "CPV": ["STP","COM","MUS"],
    "CUB": ["JAM","DOM","PRI"],
    "CUW": ["ABW","MHL"],
    "CXR": ["CCK","NFK","HMD"],
    "CYM": ["VGB","TCA","BMU"],
    "DJI": ["ERI","SOM","YEM"],
    "DMA": ["KNA","VCT","LCA"],
    "DZA": ["MAR","TUN","LBY"],
    "ERI": ["DJI","SOM","SDN"],
    "ESH": ["MAR","MRT","DZA"],
    "FLK": ["SGS","SHN","BVT"],
    "FRO": ["ISL","GRL"],
    "FSM": ["MHL","KIR","PLW"],
    "GAB": ["GNQ","COG","AGO"],
    "GGY": ["JEY","IMN","BMU"],
    "GIB": ["MLT","AND","LIE"],
    "GLP": ["MTQ","MAF","BLM"],
    "GNQ": ["GAB","COG","STP"],
    "GRL": ["ISL","FRO"],
    "GUF": ["SUR","GUY","MTQ"],
    "GUM": ["MNP","ASM","PLW"],
    "HKG": ["MAC","SGP","CHN"],
    "HMD": ["ATF","BVT","SGS"],
    "HTI": ["NIC","JAM","HND"],
    "IMN": ["CHI", "GBR"],
    "IOT": ["HMD","CCK","CXR"],
    "JAM": ["BRB","TTO","BHS"],
    "JEY": ["GGY","IMN","BMU"],
    "KAZ": ["UZB","TKM","AZE"],
    "KOR": ["JPN","CHN"],
    "KNA": ["ATG","DMA","VCT"],
    "KWT": ["QAT","BHR","OMN"],
    "LBY": ["DZA","TUN","EGY"],
    "LCA": ["VCT","DMA","ATG"],
    "LIE": ["CYP","SMR","MCO"],
    "MAC": ["HKG","SGP","CHN"],
    "MAF": ["SXM","MDG"],
    "MAR": ["TUN","DZA","EGY"],
    "MCO": ["CYP","LIE","SMR"],
    "MDA": ["UKR","GEO","ALB"],
    "MLT": ["CYP","MNE","ISL"],
    "MNP": ["GUM","ASM","PLW"],
    "MRT": ["TCD","NER"],
    "MTQ": ["GLP","MAF","BLM"],
    "MWI": ["MOZ","ZMB","TZA"],
    "MYS": ["THA","IDN","VNM"],
    "MYT": ["REU","COM","MUS"],
    "NCL": ["WSM"],
    "NFK": ["CCK","CXR","HMD"],
    "NIC": ["HND","GTM","SLV"],
    "NZL": ["AUS"],
    "OMN": ["QAT","ARE","BHR"],
    "PCN": ["TKL","NIU","NFK"],
    "PRI": ["PAN","TTO","JAM"],
    "PRK": ["VNM","LAO","MMR"],
    "PRY": ["BOL","PER","URY"],
    "PYF": ["WSM"],
    "QAT": ["KWT","BHR","ARE"],
    "REU": ["MYT","MUS","COM"],
    "SAU": ["ARE","QAT","ARE"],
    "SGS": ["FLK","BVT","ATF"],
    "SHN": ["FLK","PCN","NFK"],
    "SJM": ["GRL","FRO","ISL"],
    "SLB": ["VUT","PNG","FJI"],
    "SMR": ["CYP","LIE","MCO"],
    "SPM": ["BMU","JEY","GGY"],
    "SSD": ["TCD","CAF","ERI"],
    "SXM": ["ABW","CUW","MAF"],
    "SYR": ["IRQ","JOR","LBN"],
    "TCA": ["CYM","VGB","ABW"],
    "TCD": ["CAF","NER","SSD"],
    "TKM": ["UZB","KAZ","AZE"],
    "TWN": ["KOR","JPN","HKG"],
    "UMI": ["PCN","NFK","CXR"],
    "UZB": ["KAZ","TKM","KGZ"],
    "VAT": ["SMR","MCO","CYP"],
    "VCT": ["LCA","DMA","ATG"],
    "VEN": ["COL","ECU","PER"],
    "VGB": ["CYM","TCA","ABW"],
    "VIR": ["VGB","ABW","CUW"],
    "YEM": ["SOM","SDN","ERI"],

}

In [ ]:
for col in ['%Indoor Essential Workers', '%Essential Workers']:
    cont = True
    while cont:
        cont = False
        for idx, row in labourForce_df.iterrows():
            if pd.isna(row[col]):
                try:
                    code = row["Country Code"]
                    ave = []
                    for iso in similar_iso3.get(code, []):  # safer: avoid KeyError
                        match = labourForce_df.loc[labourForce_df["Country Code"] == iso, col]
                        if not match.empty:
                            z = match.iloc[0]   # safely take the first value
                            if not pd.isna(z):
                                ave.append(float(z))
                    if ave:  
                        # update if we found at least one valid value
                        labourForce_df.at[idx, col] = sum(ave) / len(ave)
                    else:
                        # couldn't fill this row --> rerun
                        cont = True
                except KeyError:
                    cont = True
                    pass


In [ ]:
for col in ['%Indoor Vital Workers', '%Vital Workers']:
    cont = True
    while cont:
        cont = False
        for idx, row in labourForce_df.iterrows():
            if pd.isna(row[col]):
                try:
                    code = row["Country Code"]
                    ave = []
                    for iso in similar_iso3.get(code, []):  # safer: avoid KeyError
                        match = labourForce_df.loc[labourForce_df["Country Code"] == iso, col]
                        if not match.empty:
                            z = match.iloc[0]   # safely take the first value
                            if not pd.isna(z):
                                ave.append(float(z))
                    if ave:  
                        # update if we found at least one valid value
                        labourForce_df.at[idx, col] = sum(ave) / len(ave)
                    else:
                        # couldn't fill this row → rerun
                        cont = True
                except KeyError:
                    cont = True
                    pass


In [ ]:
missing_code_list = []
missing_country_list = []
for idx, row in labourForce_df.iterrows():
    country = row["Country Name"]
    code = row["Country Code"]
    if pd.isna(row['%Indoor Essential Workers']):
      missing_code_list.append(code)
      missing_country_list.append(country)
    if pd.isna(row['%Indoor Vital Workers']):
      missing_code_list.append(code)
      missing_country_list.append(country)
    if pd.isna(row['%Essential Workers']):
      missing_code_list.append(code)
      missing_country_list.append(country)
    if pd.isna(row['%Vital Workers']):
      missing_code_list.append(code)
      missing_country_list.append(country)
if missing_country_list or missing_code_list:
    print('Warning: missing country data')
    print(missing_country_list)

In [ ]:
labourForce_df['Indoor Essential Workers'] = (labourForce_df['%Indoor Essential Workers'] * labourForce_df['Labour Force (2024)'])
labourForce_df['Indoor Vital Workers'] = (labourForce_df['%Indoor Vital Workers'] * labourForce_df['Labour Force (2024)'])
labourForce_df['Essential Workers'] = (labourForce_df['%Essential Workers'] * labourForce_df['Labour Force (2024)'])
labourForce_df['Vital Workers'] = (labourForce_df['%Vital Workers'] * labourForce_df['Labour Force (2024)'])
labourForce_df['Armed Forces (Indoor Essential)'] = (labourForce_df['%Armed Forces (Indoor Essential)'] * labourForce_df['Labour Force (2024)'])
labourForce_df['Armed Forces (Essential)'] = (labourForce_df['%Armed Forces (Essential)'] * labourForce_df['Labour Force (2024)'])

In [ ]:
print(f"{labourForce_df['Indoor Essential Workers'].sum(skipna=True):.2e}")

8.19e+08


In [ ]:
pd.set_option('display.max_rows', None)

In [ ]:
labourForce_df

,Country Name,Country Code,Labour Force (2024),Region,%Indoor Essential Workers,%Indoor Vital Workers,%Essential Workers,%Vital Workers,%Armed Forces (Indoor Essential),%Armed Forces (Essential),Indoor Essential Workers,Indoor Vital Workers,Essential Workers,Vital Workers,Armed Forces (Indoor Essential),Armed Forces (Essential)
0,Aruba,ABW,5.582600e+04,Caribbean,0.206426,0.087383,0.420866,0.239804,NaN,NaN,1.152395e+04,4.878238e+03,2.349527e+04,1.338731e+04,NaN,NaN
1,Afghanistan,AFG,9.130000e+06,Southern Asia,0.188936,0.079633,0.687096,0.530522,0.019308,0.019308,1.724990e+06,7.270474e+05,6.273188e+06,4.843667e+06,176284.793464,176284.793464
2,Angola,AGO,1.600000e+07,Middle Africa,0.202634,0.053656,0.734060,0.531812,0.004678,0.004678,3.242149e+06,8.584902e+05,1.174496e+07,8.508994e+06,74843.878585,74843.878585
3,Albania,ALB,1.370000e+06,Southern Europe,0.219481,0.105186,0.552770,0.380994,0.001667,0.001667,3.006895e+05,1.441042e+05,7.572944e+05,5.219623e+05,2283.547149,2283.547149
4,Andorra,AND,5.050400e+04,Southern Europe,0.188693,0.059175,0.317204,0.122881,NaN,NaN,9.529746e+03,2.988598e+03,1.602005e+04,6.205986e+03,NaN,NaN
5,United Arab Emirates,ARE,7.090000e+06,Western Asia,0.168646,0.058844,0.335069,0.145750,0.002412,0.002412,1.195698e+06,4.172053e+05,2.375636e+06,1.033371e+06,17101.379525,17101.379525
6,Argentina,ARG,2.230000e+07,South America,0.236126,0.065460,0.387883,0.136329,0.000000,0.000000,5.265604e+06,1.459756e+06,8.649783e+06,3.040126e+06,0.000000,0.000000
7,Armenia,ARM,1.510000e+06,Western Asia,0.196848,0.096302,0.570992,0.421732,NaN,NaN,2.972403e+05,1.454157e+05,8.621979e+05,6.368149e+05,NaN,NaN
8,American Samoa,ASM,5.595800e+04,Polynesia,0.192788,0.079255,0.554787,0.387975,NaN,NaN,1.078804e+04,4.434937e+03,3.104477e+04,2.171033e+04,NaN,NaN
9,Antigua and Barbuda,ATG,4.654000e+04,Caribbean,0.178819,0.117978,0.757144,0.664737,NaN,NaN,8.322253e+03,5.490694e+03,3.523748e+04,3.093688e+04,NaN,NaN


# Final Result
###### This table is the relevant information we will be using as the total number of essential / indoor workers in a given region. This contains both a lower and an upper bound. The lower bound is sourced by an internal census between our team and the project hosts, ALLFED, to evaluate the cirticallity of ISCO-08 Level 4 codes. The Upper bound is similarly derived however the essential ISCO-08 Level 2 job codes were pulled from an ILO study on the effects of COVID-19.

In [ ]:
regional_labourForce_df = labourForce_df.groupby('Region')[['Labour Force (2024)','Indoor Essential Workers', 'Indoor Vital Workers', 'Essential Workers', 'Vital Workers', 'Armed Forces (Indoor Essential)', 'Armed Forces (Essential)']].sum().reset_index()
regional_labourForce_df['%Indoor Essential Workers'] = regional_labourForce_df['Indoor Essential Workers']/regional_labourForce_df['Labour Force (2024)']
regional_labourForce_df['%Indoor Vital Workers'] = regional_labourForce_df['Indoor Vital Workers']/regional_labourForce_df['Labour Force (2024)']
regional_labourForce_df['%Essential Workers'] = regional_labourForce_df['Essential Workers']/regional_labourForce_df['Labour Force (2024)']
regional_labourForce_df['%Vital Workers'] = regional_labourForce_df['Vital Workers']/regional_labourForce_df['Labour Force (2024)']
regional_labourForce_df['%Armed Forces (Indoor Essential)'] = regional_labourForce_df['Armed Forces (Indoor Essential)']/regional_labourForce_df['Labour Force (2024)']
regional_labourForce_df['%Armed Forces (Essential)'] = regional_labourForce_df['Armed Forces (Essential)']/regional_labourForce_df['Labour Force (2024)']


display(regional_labourForce_df)

,Region,Labour Force (2024),Indoor Essential Workers,Indoor Vital Workers,Essential Workers,Vital Workers,Armed Forces (Indoor Essential),Armed Forces (Essential),%Indoor Essential Workers,%Indoor Vital Workers,%Essential Workers,%Vital Workers,%Armed Forces (Indoor Essential),%Armed Forces (Essential)
0,Australia and New Zealand,1.802000e+07,3.479462e+06,1.372191e+06,5.747124e+06,2.681942e+06,2345.649214,2345.649214,0.193089,0.076148,0.318930,0.148831,0.000130,0.000130
1,Caribbean,1.989858e+07,4.231121e+06,1.390071e+06,8.288341e+06,4.018966e+06,17050.518855,17050.518855,0.212634,0.069858,0.416529,0.201973,0.000857,0.000857
2,Central America,8.375000e+07,2.107288e+07,6.385517e+06,4.165108e+07,1.980532e+07,48469.372257,48469.372257,0.251616,0.076245,0.497326,0.236481,0.000579,0.000579
3,Central Asia,3.272000e+07,6.269075e+06,2.966177e+06,1.849502e+07,1.338426e+07,5719.945979,5719.945979,0.191598,0.090653,0.565251,0.409054,0.000175,0.000175
4,Eastern Africa,2.116690e+08,3.793052e+07,1.972793e+07,1.520655e+08,1.237637e+08,205581.981220,205581.981220,0.179197,0.093202,0.718412,0.584704,0.000971,0.000971
5,Eastern Asia,8.963720e+08,1.978698e+08,8.645199e+07,4.442444e+08,2.767872e+08,3146.976207,3146.976207,0.220745,0.096447,0.495603,0.308786,0.000004,0.000004
6,Eastern Europe,1.465706e+08,2.808300e+07,1.245696e+07,5.481879e+07,3.065141e+07,113411.587704,113411.587704,0.191601,0.084989,0.374009,0.209124,0.000774,0.000774
7,Melanesia,4.733000e+06,1.245589e+06,4.765797e+05,3.136924e+06,2.048281e+06,2781.961296,2781.961296,0.263171,0.100693,0.662777,0.432766,0.000588,0.000588
8,Micronesia,1.778560e+05,3.600706e+04,1.401059e+04,8.206178e+04,4.852754e+04,61.723348,61.723348,0.202451,0.078775,0.461394,0.272847,0.000347,0.000347
9,Middle Africa,7.833350e+07,1.450305e+07,7.198985e+06,5.374187e+07,4.365786e+07,80173.274738,80173.274738,0.185145,0.091902,0.686065,0.557333,0.001023,0.001023


In [ ]:
results_folder = cwd.parent / "results"
output_file_region = results_folder / "EssentialWorkersByRegion.csv"
regional_labourForce_df.to_csv(output_file_region, index=False)

In [ ]:
output_file_country = results_folder / "EssentialWorkersByCountry.csv"
labourForce_df.to_csv(output_file_country, index=False)

In [ ]:
# Validation: compare our per-country %Essential Workers to the ILO 2023
# published per-country "Share of key workers" (intersection of essential
# ISCO occupations and essential ISIC sectors). Source:
#   ILO (2023). WESO: The value of essential work.
#   https://www.ilo.org/sites/default/files/wcmsp5/groups/public/@dgreports/@dcomm/@publ/documents/publication/wcms_871016.pdf

ilo_pct_PATH = data_folder / "ILO_country_essential_workers_pct.xlsx"
ilo_pct_df = pd.read_excel(ilo_pct_PATH, sheet_name='Sheet1', header=1, engine='openpyxl')
ilo_pct_df = ilo_pct_df.rename(columns={
    'cname': 'Country Name',
    'Share of key workers': 'ILO %essential (published)',
    'Same share without agriculture': 'ILO %essential non-agri (published)',
})
ilo_pct_df = ilo_pct_df[['Country Name', 'ILO %essential (published)', 'ILO %essential non-agri (published)']]
ilo_pct_df = ilo_pct_df[
    ilo_pct_df['Country Name'].notna() & (ilo_pct_df['Country Name'] != 'Average')
].copy()
ilo_pct_df['Country Name'] = cc.convert(
    ilo_pct_df['Country Name'].tolist(), to='name_short', not_found='not found'
)
ilo_pct_df['ILO %essential (published)'] = pd.to_numeric(
    ilo_pct_df['ILO %essential (published)'], errors='coerce'
)
ilo_pct_df['ILO %essential non-agri (published)'] = pd.to_numeric(
    ilo_pct_df['ILO %essential non-agri (published)'], errors='coerce'
)

val_df = labourForce_df.merge(
    ilo_pct_df, on='Country Name', how='inner'
)
val_df['Our %Essential (pct)'] = val_df['%Essential Workers'] * 100
val_df['Delta (pp)'] = val_df['Our %Essential (pct)'] - val_df['ILO %essential (published)']

global_essential = labourForce_df['Essential Workers'].sum(skipna=True)
global_vital = labourForce_df['Vital Workers'].sum(skipna=True)
global_indoor_essential = labourForce_df['Indoor Essential Workers'].sum(skipna=True)
global_indoor_vital = labourForce_df['Indoor Vital Workers'].sum(skipna=True)
global_lf = labourForce_df['Labour Force (2024)'].sum(skipna=True)
af_global_essential = labourForce_df['Armed Forces (Essential)'].sum(skipna=True)
af_global_indoor_essential = labourForce_df['Armed Forces (Indoor Essential)'].sum(skipna=True)

print(f"Countries matched against ILO published data: {len(val_df)}")
print()
print(f"Global Labour Force (WB 2024):                  {global_lf:.3e}")
print(f"Global Essential Workers (ours):                 {global_essential:.3e} ({100*global_essential/global_lf:.2f}% of LF)")
print(f"Global Vital Workers (ours):                    {global_vital:.3e} ({100*global_vital/global_lf:.2f}% of LF)")
print(f"Global Indoor Essential Workers (ours):          {global_indoor_essential:.3e} ({100*global_indoor_essential/global_lf:.2f}% of LF)")
print(f"Global Indoor Vital Workers (ours):             {global_indoor_vital:.3e} ({100*global_indoor_vital/global_lf:.2f}% of LF)")
print()
print(f"Our mean %Essential (matched 90 countries):      {val_df['Our %Essential (pct)'].mean():.2f}%")
print(f"ILO published mean %essential (matched):        {val_df['ILO %essential (published)'].mean():.2f}%  (paper average 51.72%)")
print(f"Mean absolute deviation (percentage points):    {val_df['Delta (pp)'].abs().mean():.2f}")
print(f"Pearson correlation (ours vs ILO):              {val_df['Our %Essential (pct)'].corr(val_df['ILO %essential (published)']):.3f}")
print()
print(f"Armed-forces share of global Essential Workers:        {af_global_essential:.3e} ({100*af_global_essential/global_essential:.2f}%)")
print(f"Armed-forces share of global Indoor Essential Workers: {af_global_indoor_essential:.3e} ({100*af_global_indoor_essential/global_indoor_essential:.2f}%)")
print()
worst = val_df.reindex(val_df['Delta (pp)'].abs().sort_values(ascending=False).index)
print('10 countries with largest |Delta|:')
print(worst[['Country Name','Our %Essential (pct)','ILO %essential (published)','Delta (pp)']].head(10).to_string(index=False))

val_out_cols = ['Country Name','Country Code','Labour Force (2024)',
                'Essential Workers','%Essential Workers','Our %Essential (pct)',
                'ILO %essential (published)','ILO %essential non-agri (published)','Delta (pp)',
                'Armed Forces (Essential)']
val_df[val_out_cols].to_csv(results_folder / 'Essential_Workers_Validation.csv', index=False)

Countries matched against ILO published data: 89

Global Labour Force (WB 2024):                  3.731e+09
Global Essential Workers (ours):                 1.946e+09 (52.16% of LF)
Global Vital Workers (ours):                    1.272e+09 (34.09% of LF)
Global Indoor Essential Workers (ours):          8.187e+08 (21.94% of LF)
Global Indoor Vital Workers (ours):             3.688e+08 (9.88% of LF)

Our mean %Essential (matched 90 countries):      51.25%
ILO published mean %essential (matched):        51.85%  (paper average 51.72%)
Mean absolute deviation (percentage points):    4.91
Pearson correlation (ours vs ILO):              0.883

Armed-forces share of global Essential Workers:        3.259e+06 (0.17%)
Armed-forces share of global Indoor Essential Workers: 3.259e+06 (0.40%)

10 countries with largest |Delta|:
         Country Name  Our %Essential (pct)  ILO %essential (published)  Delta (pp)
              Liberia             40.292200                       68.48  -28.187800
     